# M06. Base Running
- This predicts errors, double plays, out locations, and baserunner advancements
- Type: Model
- Run Frequency: Irregular
- Sources:
    - MLB API
    - Steamer
- Created: 12/16/2023
- Updated: 2/5/2025

## Imports

In [ ]:
%run "U01. Imports.ipynb"
%run "U02. Functions.ipynb"
%run "U03. Classes.ipynb"
%run "U04. Datasets.ipynb"

In [ ]:
pd.set_option('display.float_format', lambda x: '%.4f' % x)

### Data

##### Steamer

In [ ]:
steamer_hitters_df = pd.read_csv(os.path.join(baseball_path, "A03. Steamer", "steamer_hitters_weekly_log.csv"), encoding='iso-8859-1')

##### MLB Stats API

Notes: 
- This cannot use the same complete dataset as elsewhere because multiple records per plate appearance are required and those are typically dropped
- MLB Stats API calls are highly prone to connection errors. Because of this, it's highly discouraged to run them all in parallel or in one large call. Sadly, the below approach is the best I've got.

Create yearly dataframes

In [ ]:
# df2015 = plays_statsapi("04/01/2015", "10/31/2015")
# df2016 = plays_statsapi("04/01/2016", "10/31/2016")
# df2017 = plays_statsapi("04/01/2017", "10/31/2017")
# df2018 = plays_statsapi("04/01/2018", "10/31/2018")
# df2019 = plays_statsapi("04/01/2019", "10/31/2019")
# df2020 = plays_statsapi("04/01/2020", "10/31/2020")
# df2021 = plays_statsapi("04/01/2021", "10/31/2021")
# df2022 = plays_statsapi("04/01/2022", "10/31/2022")
# df2023 = plays_statsapi("04/01/2023", "10/31/2023")
# df2024 = plays_statsapi("04/01/2024", "10/31/2024")
# df2025 = plays_statsapi("04/01/2025", "11/01/2025")

Concatenate yearly dataframes

In [ ]:
# running_dataset = pd.concat([df2015, df2016, df2017, df2018, df2019, df2020, df2021, df2022, df2023, df2024, df2025], axis=0).query('game_type == "R"')

Write to CSV

In [ ]:
# running_dataset.to_csv(os.path.join(baseball_path, "Running Dataset.csv"), index=False)

Read CSV

In [ ]:
running_dataset = pd.read_csv(os.path.join(baseball_path, "Running Dataset.csv"))

### Clean

##### Steamer

Create derived statistics and drop records with missing data

In [ ]:
steamer_hitters_df2 = clean_steamer_hitters(steamer_hitters_df).dropna(subset=batter_stats_fg)

Calculate imputed stolen base attempt and success rates

In [ ]:
steamer_hitters_df2['sba_imp'] = steamer_hitters_df2['sba'] / steamer_hitters_df2['sbo']
steamer_hitters_df2['sb_imp'] = steamer_hitters_df2['sb'] / steamer_hitters_df2['sba']

Format data types for merge

In [ ]:
steamer_hitters_df2['date'] = steamer_hitters_df2['date'].astype(int)
steamer_hitters_df2['mlbamid'] = steamer_hitters_df2['mlbamid'].astype(int)
steamer_hitters_df2['date'] = pd.to_datetime(steamer_hitters_df2['date'], format='%Y%m%d')

##### MLB Stats API

There are two major issues with the MLB Stats API data that require correcting before use in training models:
1. Base runners that move multiple bases during a plate appearance will not have their full movement recorded within a single observation and may be treated as duplicates
2. Base runners that do not move during a plate appearance will not have their own observation in the data <br>

Since the model requires knowing where base runners/batters start and end with each plate appearance, we need to address these problems.

##### Problem 1. Multiple Movements

Identify where a runner starts and ends in an at bat. Only keep one instance. <br>
Note: This step needs to be done first to avoid creating duplicate observations.

##### Determine start and end bases

Assumption: runner on base after the previous PA is the runner on base at the start of the next PA

Extract id for base runners after the PA

In [ ]:
running_dataset['postOnFirst'] = running_dataset['postOnFirst'].str.extract(r"'id': (\d+)")
running_dataset['postOnSecond'] = running_dataset['postOnSecond'].str.extract(r"'id': (\d+)")
running_dataset['postOnThird'] = running_dataset['postOnThird'].str.extract(r"'id': (\d+)")

Shift by one to apply the plate appearances end state onto the next observation's start state

In [ ]:
running_dataset['preOnFirst'] = (running_dataset.groupby(['gamePk', 'inning', 'halfInning'])['postOnFirst'].shift(1))
running_dataset['preOnSecond'] = (running_dataset.groupby(['gamePk', 'inning', 'halfInning'])['postOnSecond'].shift(1))
running_dataset['preOnThird'] = (running_dataset.groupby(['gamePk', 'inning', 'halfInning'])['postOnThird'].shift(1))

Fill missings with "Empty" - this allows for the subsequent transform('first') column to accept missings as a first value

In [ ]:
running_dataset[['preOnFirst', 'preOnSecond', 'preOnThird']] = running_dataset[['preOnFirst', 'preOnSecond', 'preOnThird']].fillna("Empty")

Assign the first observation's base start state's to the entire PA

In [ ]:
running_dataset[['preOnFirst', 'preOnSecond', 'preOnThird']] = (running_dataset.groupby(['gamePk', 'inning', 'halfInning', 'atBatIndex'])[['preOnFirst', 'preOnSecond', 'preOnThird']].transform('first'))

Determine start and end base by number:
- 0. AB
- 1. 1B
- 2. 2B
- 3. 3B
- 4. Scored
- 5. Out

In [ ]:
running_dataset['startInt'] = running_dataset['start'].apply(lambda x: 0 if pd.isna(x) else int(x[0]) if x[0].isdigit() else 0)
running_dataset['endInt'] = running_dataset['end'].apply(lambda x: 5 if pd.isna(x) else 4 if x.lower() == 'score' else int(x[0]) if x[0].isdigit() else 0)

In [ ]:
running_dataset['minBase'] = running_dataset.groupby(['gamePk', 'atBatIndex', 'runner_id'])['startInt'].transform('min')
running_dataset['maxBase'] = running_dataset.groupby(['gamePk', 'atBatIndex', 'runner_id'])['endInt'].transform('max')

Notes:
- At this state, we have the start (minBase) and end (maxBase) locations for all batters/base runners **who moved** (inclusive of batters who got out in any fashion).
- If a base runner moved for multiple reasons (let's say from first to second on a single and second to third on an error), they may appear in the dataset multiple times.

Warning: minBase and maxBase are used to determine how PA events affect base runners. However, they can also change via stolen base. If this is not accounted for, there may exist higher than expected advance probabilities. For example, a runner stealing second and then scoring on a single would have a minBase of 1 and a maxBase of 4, but attributing three bases of movement to the single would be unwise. This can be addressed in two possible ways:
1. Adjusting minBase and maxBase to account for the states right before the deciding pitch, not at the start of the PA
2. Removing such PAs from the sample. There's little downside to this as samples are large enough after removing PAs with SBs, so let's do this for now! PAs with stolen base attempts and other non-typical advances will be excluded using the movementTypical variable.

Identify plate appearances with only typical movement (to be included in base running models)

Movement Reasons:
- r_adv_force: advanced on a ball in play because they were forced to
- r_adv_play: advanced on a ball in play without being forced to
- r_force_out: out on a force play
- r_adv_throw: advanced on the throw, not the contact
- r_runner_out: out not on a force play
- r_thrown_out: out on a hit (base runner)
- r_doubled_off: out on a ball caught and thrown to base
- r_out_stretching: out on a hit (hitter)

In [ ]:
movementReason_list = ['r_adv_force', 'r_adv_play', 'r_force_out', 'r_adv_throw', 'r_runner_out', 'r_thrown_out', 'r_doubled_off', 'r_out_stretching']

running_dataset['movementTypical'] = running_dataset['movementReason'].apply(lambda x: 1 if x in movementReason_list or pd.isna(x) else 0)
running_dataset['movementTypical'] = running_dataset.groupby(['gamePk', 'atBatIndex'])['movementTypical'].transform('min')

Stolen Bases and Attempts

In [ ]:
running_dataset['sb_2b'] = running_dataset['movementReason'].isin(['r_stolen_base_2b']).astype('int') 
running_dataset['sb_3b'] = running_dataset['movementReason'].isin(['r_stolen_base_3b']).astype('int') 
running_dataset['sba_2b'] = running_dataset['movementReason'].isin(['r_stolen_base_2b', 'r_caught_stealing_2b', 'r_pickoff_caught_stealing_2b']).astype('int') 
running_dataset['sba_3b'] = running_dataset['movementReason'].isin(['r_stolen_base_3b', 'r_caught_stealing_3b', 'r_pickoff_caught_stealing_3b']).astype('int') 

In [ ]:
running_dataset[['sb_2b', 'sb_3b', 'sba_2b', 'sba_3b']] = running_dataset.groupby(['gamePk', 'atBatIndex', 'runner_id'])[['sb_2b', 'sb_3b', 'sba_2b', 'sba_3b']].transform('max')

Now that movementTypical and stolen base fields are applied to entire PAs and runners, respectively, we have everything we need for base runners on each observation, allowing us to drop duplicates

In [ ]:
running_dataset.drop_duplicates(subset=['gamePk', 'atBatIndex', 'runner_id', 'startInt'], inplace=True, keep='first')

##### Problem 2. Missing Base Runners

##### At Bat

There will always be an observation for the guy at bat unless a base runner ended the inning with an out. We don't need those observations anyway.

In [ ]:
atBat = running_dataset.query('id == batter')

atBat.drop_duplicates(['gamePk', 'atBatIndex', 'runner_id'], keep='first', inplace=True)

##### Runners on 1B

Runner will either have an observation if they moved or one will be created by modifying the batter or another base runner's observation

In [ ]:
# Identify all observations with runners on specified base
on1B = running_dataset[running_dataset['preOnFirst'] != "Empty"]

# Identify observations where we have the base runner as their own observation
on1B['is_runner'] = (on1B['preOnFirst'].astype(str) == on1B['runner_id'].astype(str)).astype(int)

# Sort to bring those observations to the top of their PA, where possible
on1B = on1B.sort_values(by=['gamePk', 'atBatIndex', 'is_runner', 'startInt'], ascending=[True, True, False, True])

# Drop duplicates, keeping one record per PA, preferring the actual base runner, where possible
on1B.drop_duplicates(subset=['gamePk', 'atBatIndex'], keep='first', inplace=True)

# Overwrite base runners to make them valid versions of the runner on base
on1B.loc[on1B['start'] != "1B", ['startInt', 'endInt', 'minBase', 'maxBase']] = 1
on1B.loc[on1B['start'] != "1B", ['start', 'end']] = "1B"
on1B.loc[on1B['start'] != "1B", ['id', 'runner_id']] = on1B.loc[on1B['start'] != "1B", 'preOnFirst'].rename("id").to_frame().assign(runner_id=lambda df: df['id'])

##### Runners on 2B

In [ ]:
# Identify all observations with runners on specified base
on2B = running_dataset[running_dataset['preOnSecond'] != "Empty"]

# Identify observations where we have the base runner as their own observation
on2B['is_runner'] = (on2B['preOnSecond'].astype(str) == on2B['runner_id'].astype(str)).astype(int)

# Sort to bring those observations to the top of their PA, where possible
on2B = on2B.sort_values(by=['gamePk', 'atBatIndex', 'is_runner', 'startInt'], ascending=[True, True, False, True])

# Drop duplicates, keeping one record per PA, preferring the actual base runner, where possible
on2B.drop_duplicates(subset=['gamePk', 'atBatIndex'], keep='first', inplace=True)

# Overwrite base runners to make them valid versions of the runner on base
on2B.loc[on2B['start'] != "2B", ['startInt', 'endInt', 'minBase', 'maxBase']] = 2
on2B.loc[on2B['start'] != "2B", ['start', 'end']] = "2B"
on2B.loc[on2B['start'] != "2B", ['id', 'runner_id']] = on2B.loc[on2B['start'] != "2B", 'preOnSecond'].rename("id").to_frame().assign(runner_id=lambda df: df['id'])

##### Runners on 3B

In [ ]:
# Identify all observations with runners on specified base
on3B = running_dataset[running_dataset['preOnThird'] != "Empty"]

# Identify observations where we have the base runner as their own observation
on3B['is_runner'] = (on3B['preOnThird'].astype(str) == on3B['runner_id'].astype(str)).astype(int)

# Sort to bring those observations to the top of their PA, where possible
on3B = on3B.sort_values(by=['gamePk', 'atBatIndex', 'is_runner', 'startInt'], ascending=[True, True, False, True])

# Drop duplicates, keeping one record per PA, preferring the actual base runner, where possible
on3B.drop_duplicates(subset=['gamePk', 'atBatIndex'], keep='first', inplace=True)

# Overwrite base runners to make them valid versions of the runner on base
on3B.loc[on3B['start'] != "3B", ['startInt', 'endInt', 'minBase', 'maxBase']] = 3
on3B.loc[on3B['start'] != "3B", ['start', 'end']] = "3B"
on3B.loc[on3B['start'] != "3B", ['id', 'runner_id']] = on3B.loc[on3B['start'] != "3B", 'preOnThird'].rename("id").to_frame().assign(runner_id=lambda df: df['id'])

##### Combine

In [ ]:
df = pd.concat([atBat, on1B, on2B, on3B], axis=0, ignore_index=True)

Sort to group PAs together

In [ ]:
df['atBatIndexNum'] = df.groupby(['gamePk', 'atBatIndex']).cumcount() + 1
df.sort_values(['gamePk', 'atBatIndex', 'atBatIndexNum'], inplace=True)

##### Base Dummies

Start Locations

In [ ]:
df['pre_1b'] = (df['preOnFirst'] != "Empty").astype('int')
df['pre_2b'] = (df['preOnSecond'] != "Empty").astype('int')
df['pre_3b'] = (df['preOnThird'] != "Empty").astype('int')

End Locations

In [ ]:
df['post_1b'] = pd.notna(df['postOnFirst']).astype('int')
df['post_2b'] = pd.notna(df['postOnSecond']).astype('int')
df['post_3b'] = pd.notna(df['postOnThird']).astype('int')

Blocked Bases (bases a given base runner could not advance to because there's a runner **ahead** of them)

In [ ]:
df['blocked_1b'] = ((df['post_1b'] == 1) & (df['maxBase'] < 1)).astype('int')
df['blocked_2b'] = ((df['post_2b'] == 1) & (df['maxBase'] < 2)).astype('int')
df['blocked_3b'] = ((df['post_3b'] == 1) & (df['maxBase'] < 3)).astype('int')

##### Events

In [ ]:
df = create_events(df)
df['eventsModelInt'] = df['eventsModel'].map({'b1': 1, 'b2': 2, 'b3': 3, 'hr': 4, 'bb': 5, 'hbp': 6, 'so': 7, 'fo': 8, 'go': 9, 'lo': 10, 'po': 11})

Create dummies

In [ ]:
event_dummies = pd.get_dummies(df[df['eventsModel'] != "Cut"]['eventsModel'], prefix='event').astype(int)
minBase_dummies = pd.get_dummies(df['minBase'], prefix='minBase').astype(int)

df = pd.concat([df, event_dummies, minBase_dummies], axis=1)

##### Out Locations

Runner is out dummy

In [ ]:
df['out'] = (df['maxBase'] == 5).astype('int')

Base out dummies

In [ ]:
df['out_home'] = ((df['out'] == 1) & (df['minBase'] == 0)).astype('int')
df['out_1b'] = ((df['out'] == 1) & (df['minBase'] == 1)).astype('int')
df['out_2b'] = ((df['out'] == 1) & (df['minBase'] == 2)).astype('int')
df['out_3b'] = ((df['out'] == 1) & (df['minBase'] == 3)).astype('int')

df['out_home'] = df.groupby(['gamePk', 'atBatIndex'])['out_home'].transform('max')
df['out_1b'] = df.groupby(['gamePk', 'atBatIndex'])['out_1b'].transform('max')
df['out_2b'] = df.groupby(['gamePk', 'atBatIndex'])['out_2b'].transform('max')
df['out_3b'] = df.groupby(['gamePk', 'atBatIndex'])['out_3b'].transform('max')

##### Outs in Inning

Calculate number of outs on play

In [ ]:
df['outs_calculated'] = df.groupby(['gamePk', 'atBatIndex'])['out'].transform('sum')

Calculate number of outs prior to PA

In [ ]:
df['outs_pre'] = df.groupby(['gamePk', 'inning', 'halfInning'])['outs'].shift(1)
df['outs_pre'] = df.groupby(['gamePk', 'atBatIndex'])['outs_pre'].transform('min')
df['outs_pre'] = np.where(df['outs_pre'] == 3, 0, df['outs_pre'])

df['outs_pre'].fillna(0, inplace=True)

##### Errors

In [ ]:
df['error'] = df['description'].fillna("Missing").str.contains('error', case=False).astype('int')
df['error'] = np.where((df['outs_calculated'] == 0) & (df['eventType'] == 'fielders_choice'), 1, df['error'])

##### Double Plays

In [ ]:
df['double_play'] = df['eventType'].isin(['grounded_into_double_play', 'double_play', 'sac_fly_double_play', 'strikeout_double_play', 'sac_bunt_double_play']).astype(int)
df['double_play'] = np.where(df['outs_calculated'] == 2, 1, df['double_play'])

##### Descriptive

Determine year

In [ ]:
df['year'] = df["game_date"].astype('str').str[:4].astype('int')

Format data types for merge

In [ ]:
df['date'] = df['game_date'].str.replace("-", "").astype(int)
df['date'] = pd.to_datetime(df['date'], format='%Y%m%d')

df['runner_id'] = df['runner_id'].astype(int)

### Merge

In [ ]:
df = pd.merge_asof(
    df.sort_values('date'),  
    steamer_hitters_df2[['mlbamid', 'date', 'sb', 'sbo', 'sba', 'sb_imp', 'sba_imp']].sort_values('date'),
    left_on='date',
    right_on='date',
    left_by='runner_id',
    right_by='mlbamid'
)

Resort in more logical order

In [ ]:
df.sort_values(['gamePk', 'atBatIndex'], inplace=True)

### Sample

Drop triple plays

In [ ]:
df = df.query('outs_calculated != 3')

Drop recent data (recent Steamer data is not available)

In [ ]:
df = df[df['year'] < 2025]

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

### Model A. Errors

##### Inputs

In [ ]:
error_input_list = list(event_dummies.columns) + ['pre_1b', 'pre_2b', 'pre_3b']

Note: removed outs_pre because of small sample sizes with specific combinations could lead to high error rates

##### Sample

Universe: Typical events. Typical base movement. Using the batter as the observation.

In [ ]:
X = df.query('eventsModel != "Cut"').query('movementTypical == 1').query('startInt == 0')[error_input_list]
y = df.query('eventsModel != "Cut"').query('movementTypical == 1').query('startInt == 0')['error']

##### Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=10)

In [ ]:
X_train = torch.tensor(X_train.to_numpy(), dtype=torch.float32, device=device)
y_train = torch.tensor(y_train.to_numpy(), dtype=torch.long, device=device)
X_test  = torch.tensor(X_test.to_numpy(), dtype=torch.float32, device=device)
y_test  = torch.tensor(y_test.to_numpy(), dtype=torch.long, device=device)

##### Settings

In [ ]:
num_classifiers = 3 # Ensemble size
random_state = random.randint(10000,90000) 

error_model_parameters = {
    'hidden_layer_sizes': (64,64),
    'activation': 'relu',
    'max_iter': 100,
    'alpha': 0.00001,
    'learning_rate_init': 0.01, 
    'batch_size': 'auto',
    'random_state': random_state,
    'early_stopping': True,
    'tol': 0.00001,
    'n_iter_no_change': 20,
    'validation_fraction': 0.05
}

##### Train

In [ ]:
# ------------------------
# SETTINGS / PARAMETERS
# ------------------------
input_size = X_train.shape[1]
output_size = 2  # binary classification
hidden_layers = error_model_parameters['hidden_layer_sizes']
lr = error_model_parameters['learning_rate_init']
num_epochs = error_model_parameters['max_iter']

num_classifiers = num_classifiers  # ensure defined
random_seed = random_state         # ensure defined

# Directory to save
save_dir = os.path.join(model_path, "M06. Base Running", todaysdate)
os.makedirs(save_dir, exist_ok=True)

error_filename = "predict_errors"

# ------------------------
# TRAIN ENSEMBLE
# ------------------------
ensemble = []

for j in range(num_classifiers):
    print(f"Training classifier {j+1}/{num_classifiers}")

    seed = random_seed + 100*j
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)

    # Build model
    model = MLP(input_size, hidden_layers, output_size).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    # Train
    model.train()
    for epoch in range(num_epochs):
        optimizer.zero_grad()
        outputs = model(X_train)
        loss = criterion(outputs, y_train)
        loss.backward()
        optimizer.step()

    ensemble.append(model)

# ------------------------
# SAVE PYTORCH ENSEMBLE (.pt)
# ------------------------
torch.save({
    'state_dicts': [m.state_dict() for m in ensemble],
    'input_size': input_size,
    'hidden_layers': hidden_layers,
    'output_size': output_size
}, os.path.join(save_dir, f'{error_filename}.pt'))

# ------------------------
# EXPORT TO NUMPY FOR FAST INFERENCE
# ------------------------
ensemble_numpy = []

for m in ensemble:
    state_dict = m.state_dict()
    layers = []

    linear_keys = [k for k in state_dict.keys() if "weight" in k]
    linear_keys.sort()

    for key in linear_keys:
        W = state_dict[key].cpu().numpy().T
        b = state_dict[key.replace("weight", "bias")].cpu().numpy()
        layers.append(W)
        layers.append(b)

    ensemble_numpy.append(layers)

# ------------------------
# BUILD NUMPY WRAPPER
# ------------------------
metadata = {
    "hidden_layers": hidden_layers,
    "num_classifiers": num_classifiers,
    "random_seed": random_seed,
    "training_epochs": num_epochs
}

predict_errors = NumpyPredict(
    ensemble_numpy=ensemble_numpy,
    input_columns=error_input_list,
    classes=[0,1],
    metadata=metadata
)

# ------------------------
# SAVE WRAPPER
# ------------------------
pickle_filename = os.path.join(save_dir, f"{error_filename}.sav")
with open(pickle_filename, "wb") as f:
    pickle.dump(predict_errors, f)

print(f"Saved NumpyPredict wrapper to {pickle_filename}")

##### Predict

In [ ]:
# Convert X_test tensor back to DataFrame
X_test_df = pd.DataFrame(X_test.cpu().numpy(), columns=error_input_list)

# Convert y_test tensor back to Series with name "error"
y_test_df = pd.Series(y_test.cpu().numpy(), name='error')

# Get probability predictions using the NumpyPredict wrapper
probabilities = predict_errors.predict_proba(X_test_df)

# Create DataFrame from probabilities
probability_columns = [f'error_{label}' for label in predict_errors.classes_]
probability_df = pd.DataFrame(probabilities, columns=probability_columns, index=X_test_df.index)

# Concatenate X_test, y_test ("error"), and probabilities
error_df = pd.concat([X_test_df, y_test_df, probability_df], axis=1)

##### Evaluate

In [ ]:
quantiles = 20

# Set Variable
var = "error"

# Copy 
graph_df = error_df.copy()

# Step 1: Divide error_1 into 
graph_df['quantile'] = pd.qcut(graph_df[f'{var}_1'], quantiles, labels=False, duplicates='drop')

# Step 2 & 3: Group by quantile and calculate average error_1 and error
grouped_quantiles = graph_df.groupby('quantile').agg({f'{var}_1': 'mean', f'{var}': 'mean'}).reset_index()

# Step 4: Plot the averages
plt.figure(figsize=(10, 6))
plt.plot(grouped_quantiles['quantile'], grouped_quantiles[f'{var}_1'], marker='o', label='Average (Projected)')
plt.plot(grouped_quantiles['quantile'], grouped_quantiles[f'{var}'], marker='x', label='Average (Actual)')
plt.xlabel('Quantile (Projected)')
plt.ylabel('Average')
plt.title(f'Average by Quantile, {var} == 1')
plt.legend()
plt.grid(True)
plt.show()


### Model B. Double Play

##### Inputs

In [ ]:
double_play_input_list = list(event_dummies.columns) + ['outs_pre', 'pre_1b', 'pre_2b', 'pre_3b']

##### Sample

Universe: Typical events. Typical base movement. Using the batter as the observation.

In [ ]:
X = df.query('eventsModel != "Cut"').query('movementTypical == 1').query('minBase == 0')[double_play_input_list] 
y = df.query('eventsModel != "Cut"').query('movementTypical == 1').query('minBase == 0')['double_play']

##### Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=10)

In [ ]:
X_train = torch.tensor(X_train.to_numpy(), dtype=torch.float32, device=device)
y_train = torch.tensor(y_train.to_numpy(), dtype=torch.long, device=device)
X_test  = torch.tensor(X_test.to_numpy(), dtype=torch.float32, device=device)
y_test  = torch.tensor(y_test.to_numpy(), dtype=torch.long, device=device)

##### Settings

In [ ]:
double_play_model_parameters = {
    'hidden_layer_sizes': (64,64),
    'activation': 'relu',
    'max_iter': 100,
    'alpha': 0.00001,
    'learning_rate_init': 0.01, 
    'batch_size': 'auto',
    'random_state': random_state,
    'early_stopping': True,
    'tol': 0.00001,
    'n_iter_no_change': 20,
    'validation_fraction': 0.05
}


##### Train

In [ ]:
# ------------------------
# SETTINGS / PARAMETERS
# ------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

input_size = X_train.shape[1]  # make sure X_train here is for double play
output_size = 2  # binary classification
hidden_layers = double_play_model_parameters['hidden_layer_sizes']
lr = double_play_model_parameters['learning_rate_init']
num_epochs = double_play_model_parameters['max_iter']

num_classifiers = num_classifiers  # ensure defined
random_seed = random_state         # ensure defined

# Directory to save
save_dir = os.path.join(model_path, "M06. Base Running", todaysdate)
os.makedirs(save_dir, exist_ok=True)

model_filename = "predict_dp"

# ------------------------
# TRAIN ENSEMBLE
# ------------------------
ensemble = []

for j in range(num_classifiers):
    print(f"Training classifier {j+1}/{num_classifiers}")

    seed = random_seed + 100*j
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)

    # Build model
    model = MLP(input_size, hidden_layers, output_size).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    # Train
    model.train()
    for epoch in range(num_epochs):
        optimizer.zero_grad()
        outputs = model(X_train)  # X_train for double play
        loss = criterion(outputs, y_train)  # y_train for double play
        loss.backward()
        optimizer.step()

    ensemble.append(model)

# ------------------------
# SAVE PYTORCH ENSEMBLE (.pt)
# ------------------------
torch_pt_filename = os.path.join(save_dir, f"{model_filename}.pt")
torch.save({
    'state_dicts': [m.state_dict() for m in ensemble],
    'input_size': input_size,
    'hidden_layers': hidden_layers,
    'output_size': output_size
}, torch_pt_filename)

print(f"Saved PyTorch ensemble to {torch_pt_filename}")


# ------------------------
# EXPORT TO NUMPY FOR FAST INFERENCE
# ------------------------
ensemble_numpy = []

for m in ensemble:
    state_dict = m.state_dict()
    layers = []

    linear_keys = [k for k in state_dict.keys() if "weight" in k]
    linear_keys.sort()

    for key in linear_keys:
        W = state_dict[key].cpu().numpy().T
        b = state_dict[key.replace("weight", "bias")].cpu().numpy()
        layers.append(W)
        layers.append(b)

    ensemble_numpy.append(layers)

# ------------------------
# BUILD NUMPY WRAPPER
# ------------------------
metadata = {
    "hidden_layers": hidden_layers,
    "num_classifiers": num_classifiers,
    "random_seed": random_seed,
    "training_epochs": num_epochs
}

predict_dp = NumpyPredict(
    ensemble_numpy=ensemble_numpy,
    input_columns=double_play_input_list,
    classes=[0,1],
    metadata=metadata
)

# ------------------------
# SAVE WRAPPER
# ------------------------
pickle_filename = os.path.join(save_dir, f"{model_filename}.sav")
with open(pickle_filename, "wb") as f:
    pickle.dump(predict_dp, f)

print(f"Saved NumpyPredict wrapper to {pickle_filename}")


##### Predict

In [ ]:
# Convert X_test tensor back to DataFrame
X_test_df = pd.DataFrame(X_test.cpu().numpy(), columns=double_play_input_list)

# Convert y_test tensor back to Series with name "double_play"
y_test_df = pd.Series(y_test.cpu().numpy(), name='double_play')

# Get probability predictions using the NumpyPredict wrapper
probabilities = predict_dp.predict_proba(X_test_df)

# Create DataFrame from probabilities
class_labels = predict_dp.classes_
probability_columns = [f'double_play_{label}' for label in class_labels]
probability_df = pd.DataFrame(probabilities, columns=probability_columns, index=X_test_df.index)

# Concatenate X_test, y_test ("double_play"), and probabilities
dp_df = pd.concat([X_test_df, y_test_df, probability_df], axis=1)

##### Evaluate

In [ ]:
print(dp_df[['double_play_1', 'double_play']].mean())

# Set Variable
var = "double_play"

# Copy 
graph_df = dp_df.copy()

# Step 1: Divide error_1 into quantiles
graph_df['quantile'] = pd.qcut(graph_df[f'{var}_1'], 40, labels=False, duplicates='drop')

# Step 2 & 3: Group by quantile and calculate average double_play_1 and double_play
grouped_quantiles = graph_df.groupby('quantile').agg({f'{var}_1': 'mean', f'{var}': 'mean'}).reset_index()

# Step 4: Plot the averages
plt.figure(figsize=(10, 6))
plt.plot(grouped_quantiles['quantile'], grouped_quantiles[f'{var}_1'], marker='o', label='Average (Projected)')
plt.plot(grouped_quantiles['quantile'], grouped_quantiles[f'{var}'], marker='x', label='Average (Actual)')
plt.xlabel('Quantile (Projected)')
plt.ylabel('Average')
plt.title(f'Average by Quantile, {var} == 1')
plt.legend()
plt.grid(True)
plt.show()

### Model C. Out Bases

##### Inputs

In [ ]:
out_input_list = list(event_dummies.columns) + list(minBase_dummies.columns) + ['outs_pre', 'pre_1b', 'pre_2b', 'pre_3b', 'error', 'double_play']

##### Sample

Universe: Typical events. Typical base movement. Batter and base runners as observations.

In [ ]:
X = df.query('eventsModel != "Cut"').query('movementTypical == 1')[out_input_list] 
y = df.query('eventsModel != "Cut"').query('movementTypical == 1')['out']

##### Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=10)

In [ ]:
X_train = torch.tensor(X_train.to_numpy(), dtype=torch.float32, device=device)
y_train = torch.tensor(y_train.to_numpy(), dtype=torch.long, device=device)
X_test  = torch.tensor(X_test.to_numpy(), dtype=torch.float32, device=device)
y_test  = torch.tensor(y_test.to_numpy(), dtype=torch.long, device=device)

##### Settings

In [ ]:
num_classifiers = 3 # Ensemble size
random_state = random.randint(10000,90000) 

out_bases_model_parameters = {
    'hidden_layer_sizes': (64,64),
    'activation': 'relu',
    'max_iter': 100,
    'alpha': 0.00001,
    'learning_rate_init': 0.01, 
    'batch_size': 'auto',
    'random_state': random_state,
    'early_stopping': True,
    'tol': 0.00001,
    'n_iter_no_change': 20,
    'validation_fraction': 0.05
}

##### Train

In [ ]:
# ------------------------
# SETTINGS / PARAMETERS
# ------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

input_size = X_train.shape[1]  # X_train for this model
output_size = 2  # binary classification
hidden_layers = out_bases_model_parameters['hidden_layer_sizes']
lr = out_bases_model_parameters['learning_rate_init']
num_epochs = out_bases_model_parameters['max_iter']

num_classifiers = num_classifiers  # ensure defined
random_seed = random_state         # ensure defined

# Directory to save
save_dir = os.path.join(model_path, "M06. Base Running", todaysdate)
os.makedirs(save_dir, exist_ok=True)

model_filename = "predict_out_bases"

# ------------------------
# TRAIN ENSEMBLE
# ------------------------
ensemble = []

for j in range(num_classifiers):
    print(f"Training classifier {j+1}/{num_classifiers}")

    seed = random_seed + 100*j
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)

    # Build model
    model = MLP(input_size, hidden_layers, output_size).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    # Train
    model.train()
    for epoch in range(num_epochs):
        optimizer.zero_grad()
        outputs = model(X_train)  # X_train for this model
        loss = criterion(outputs, y_train)  # y_train for this model
        loss.backward()
        optimizer.step()

    ensemble.append(model)

# ------------------------
# SAVE PYTORCH ENSEMBLE (.pt)
# ------------------------
torch.save({
    'state_dicts': [m.state_dict() for m in ensemble],
    'input_size': input_size,
    'hidden_layers': hidden_layers,
    'output_size': output_size
}, os.path.join(save_dir, f"{model_filename}.pt"))

# ------------------------
# EXPORT TO NUMPY FOR FAST INFERENCE
# ------------------------
ensemble_numpy = []

for m in ensemble:
    state_dict = m.state_dict()
    layers = []

    linear_keys = [k for k in state_dict.keys() if "weight" in k]
    linear_keys.sort()

    for key in linear_keys:
        W = state_dict[key].cpu().numpy().T
        b = state_dict[key.replace("weight", "bias")].cpu().numpy()
        layers.append(W)
        layers.append(b)

    ensemble_numpy.append(layers)

# ------------------------
# BUILD NUMPY WRAPPER
# ------------------------
metadata = {
    "hidden_layers": hidden_layers,
    "num_classifiers": num_classifiers,
    "random_seed": random_seed,
    "training_epochs": num_epochs
}

predict_out_bases = NumpyPredict(
    ensemble_numpy=ensemble_numpy,
    input_columns=out_input_list,
    classes=[0,1],
    metadata=metadata
)

# ------------------------
# SAVE WRAPPER
# ------------------------
pickle_filename = os.path.join(save_dir, f"{model_filename}.sav")
with open(pickle_filename, "wb") as f:
    pickle.dump(predict_out_bases, f)

print(f"Saved NumpyPredict wrapper to {pickle_filename}")


##### Predict

In [ ]:
# Convert X_test tensor back to DataFrame
X_test_df = pd.DataFrame(X_test.cpu().numpy(), columns=out_input_list)

# Convert y_test tensor back to Series with name "out"
y_test_df = pd.Series(y_test.cpu().numpy(), name='out')

# Get probability predictions using the NumpyPredict wrapper
probabilities = predict_out_bases.predict_proba(X_test_df)

# Create DataFrame from probabilities
probability_columns = [f'out_{label}' for label in predict_out_bases.classes_]
probability_df = pd.DataFrame(probabilities, columns=probability_columns, index=X_test_df.index)

# Concatenate X_test, y_test ("out"), and probabilities
outs_df = pd.concat([X_test_df, y_test_df, probability_df], axis=1)

##### Evaluate

In [ ]:
# Set Variable
var = "out"

# Copy 
graph_df = outs_df.copy()

# Step 1: Divide error_1 into quantiles
graph_df['quantile'] = pd.qcut(graph_df[f'{var}_1'], 40, labels=False, duplicates='drop')

# Step 2 & 3: Group by quantile and calculate average out_1 and out
grouped_quantiles = graph_df.groupby('quantile').agg({f'{var}_1': 'mean', f'{var}': 'mean'}).reset_index()

# Step 4: Plot the averages
plt.figure(figsize=(10, 6))
plt.plot(grouped_quantiles['quantile'], grouped_quantiles[f'{var}_1'], marker='o', label='Average (Projected)')
plt.plot(grouped_quantiles['quantile'], grouped_quantiles[f'{var}'], marker='x', label='Average (Actual)')
plt.xlabel('Quantile (Projected)')
plt.ylabel('Average')
plt.title(f'Average by Quantile, {var} == 1')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
outs_df.query('(event_b1 == 1 or event_b2 == 1 or event_b3 == 1 or event_bb == 1 or event_hbp == 1)')[['out', 'out_1']].mean()

In [ ]:
outs_df.query('minBase_0 == 1')[['out', 'out_1']].mean()

In [ ]:
outs_df[['out', 'out_1']].mean()

### Model D. Advances

##### Inputs

In [ ]:
advances_input_list = (list(event_dummies.columns) + list(minBase_dummies.columns) + 
                    ['outs_pre', 'pre_1b', 'pre_2b', 'pre_3b', 'blocked_1b', 'blocked_2b', 'blocked_3b', 'out_home', 'out_1b', 'out_2b', 'out_3b', 'error', 'double_play'])

In [ ]:
df.query('eventsModel != "Cut"').query('movementTypical == 1').query('out == 0').query('event_b1 == 1').query('minBase_0 == 1')[advances_input_list].head(10)

##### Sample

Universe: Typical events. Typical base movement. Batter and base runners as observations. Player was not out.

In [ ]:
X = df.query('eventsModel != "Cut"').query('movementTypical == 1').query('out == 0')[advances_input_list] 
y = df.query('eventsModel != "Cut"').query('movementTypical == 1').query('out == 0')['maxBase']

##### Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=10)

In [ ]:
X_train = torch.tensor(X_train.to_numpy(), dtype=torch.float32, device=device)
y_train = torch.tensor(y_train.to_numpy(), dtype=torch.long, device=device)
X_test  = torch.tensor(X_test.to_numpy(), dtype=torch.float32, device=device)
y_test  = torch.tensor(y_test.to_numpy(), dtype=torch.long, device=device)

##### Settings

In [ ]:
num_classifiers = 3 # Ensemble size
random_state = random.randint(10000,90000) 

advances_model_parameters = {
    'hidden_layer_sizes': (64,32,16),
    'activation': 'relu',
    'max_iter': 100,
    'alpha': 0.00001,
    'learning_rate_init': 0.01, 
    'batch_size': 'auto',
    'random_state': random_state,
    'early_stopping': True,
    'tol': 0.00001,
    'n_iter_no_change': 20,
    'validation_fraction': 0.05
}

##### Train

In [ ]:
# ------------------------
# SETTINGS / PARAMETERS
# ------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Encode numeric categories to sequential integers for PyTorch
le_advances = LabelEncoder()
y_train_encoded = le_advances.fit_transform(y_train.cpu().numpy())
y_train_tensor = torch.tensor(y_train_encoded, dtype=torch.long, device=device)

input_size = X_train.shape[1]  # features for advances model
output_size = len(le_advances.classes_)  # number of classes
hidden_layers = advances_model_parameters['hidden_layer_sizes']
lr = advances_model_parameters['learning_rate_init']
num_epochs = advances_model_parameters['max_iter']

num_classifiers = num_classifiers  # ensure defined
random_seed = random_state         # ensure defined

# Directory to save
save_dir = os.path.join(model_path, "M06. Base Running", todaysdate)
os.makedirs(save_dir, exist_ok=True)

model_filename = "predict_advances"

# ------------------------
# TRAIN ENSEMBLE
# ------------------------
ensemble = []

for j in range(num_classifiers):
    print(f"Training classifier {j+1}/{num_classifiers}")

    seed = random_seed + 100*j
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)

    # Build model
    model = MLP(input_size, hidden_layers, output_size).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    # Train
    model.train()
    for epoch in range(num_epochs):
        optimizer.zero_grad()
        outputs = model(X_train)
        loss = criterion(outputs, y_train_tensor)
        loss.backward()
        optimizer.step()

    ensemble.append(model)

# ------------------------
# SAVE PYTORCH ENSEMBLE (.pt)
# ------------------------
torch.save({
    'state_dicts': [m.state_dict() for m in ensemble],
    'input_size': input_size,
    'hidden_layers': hidden_layers,
    'output_size': output_size
}, os.path.join(save_dir, f"{model_filename}.pt"))

# ------------------------
# EXPORT TO NUMPY FOR FAST INFERENCE
# ------------------------
ensemble_numpy = []

for m in ensemble:
    state_dict = m.state_dict()
    layers = []

    linear_keys = [k for k in state_dict.keys() if "weight" in k]
    linear_keys.sort()

    for key in linear_keys:
        W = state_dict[key].cpu().numpy().T
        b = state_dict[key.replace("weight", "bias")].cpu().numpy()
        layers.append(W)
        layers.append(b)

    ensemble_numpy.append(layers)

# ------------------------
# BUILD NUMPY WRAPPER
# ------------------------
metadata = {
    "hidden_layers": hidden_layers,
    "num_classifiers": num_classifiers,
    "random_seed": random_seed,
    "training_epochs": num_epochs
}

predict_advances = NumpyPredict(
    ensemble_numpy=ensemble_numpy,
    input_columns=advances_input_list,
    classes=le_advances.classes_.tolist(),  # keep original numeric labels
    metadata=metadata
)

# ------------------------
# SAVE WRAPPER
# ------------------------
pickle_filename = os.path.join(save_dir, f"{model_filename}.sav")
with open(pickle_filename, "wb") as f:
    pickle.dump(predict_advances, f)

print(f"Saved NumpyPredict wrapper to {pickle_filename}")


##### Predict

In [ ]:
# Convert tensor to DataFrame
X_test_df = pd.DataFrame(X_test.cpu().numpy(), columns=advances_input_list)

# Preserve the original maxBase column from y_test (assuming y_test tensor corresponds to maxBase)
y_test_series = pd.Series(y_test.cpu().numpy(), name='maxBase', index=X_test_df.index)

# Get probability predictions from NumpyPredict wrapper
probabilities = predict_advances.predict_proba(X_test_df)

# Create DataFrame for probabilities
probability_columns = [f'base_{label}' for label in predict_advances.classes_]
probability_df = pd.DataFrame(probabilities, columns=probability_columns, index=X_test_df.index)

# Concatenate features, maxBase, and predicted probabilities
advances_df = pd.concat([X_test_df, y_test_series, probability_df], axis=1)

In [ ]:
advances_df.head()

In [ ]:
# Set Variables
var = "base"
origin_base = 0
destination_base = 1
event = "b1"
# This one is a little confusing, but it basically shows the probability of ending up at a given base by event type.

# Copy 
graph_df = advances_df.copy()
graph_df = advances_df.query(f'event_{event} == 1 and minBase_{origin_base} == 1')
graph_df['base_dummy'] = (graph_df['maxBase'] == destination_base).astype('int')

# Step 1: Divide variable into quantile
graph_df['quantile'] = pd.qcut(graph_df[f'{var}_{destination_base}'], 10, labels=False, duplicates='drop')

# Step 2 & 3: Group by quantiles and calculate predicted and actual destination base rates
grouped_quantiles = graph_df.groupby('quantile').agg({f'{var}_{destination_base}': 'mean', 'base_dummy': 'mean'}).reset_index()

# Step 4: Plot the averages
plt.figure(figsize=(10, 6))
plt.plot(grouped_quantiles['quantile'], grouped_quantiles[f'{var}_{destination_base}'], marker='o', label='Average (Projected)')
plt.plot(grouped_quantiles['quantile'], grouped_quantiles[f'base_dummy'], marker='x', label='Average (Actual)')
plt.xlabel('Quantile (Projected)')
plt.ylabel('Average')
plt.title(f'Average by Quantile, {var} == {origin_base} -> {destination_base}')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Define configurations for the 2x2 grid
configs = [
    {"origin_base": 1, "destination_base": 3, "event": "b1"},
    {"origin_base": 2, "destination_base": 4, "event": "b1"},
    {"origin_base": 1, "destination_base": 4, "event": "b2"},
    {"origin_base": 3, "destination_base": 4, "event": "fo"},
]

# Create a 2x2 subplot
fig, axes = plt.subplots(2, 2, figsize=(8, 8))
axes = axes.ravel()  # Flatten the 2D array of axes for easier indexing

# Iterate through configurations and axes
for i, config in enumerate(configs):
    origin_base = config["origin_base"]
    destination_base = config["destination_base"]
    event = config["event"]
    var = "base"

    # Copy and filter data
    graph_df = advances_df.query(f'event_{event} == 1 and minBase_{origin_base} == 1')
    graph_df['base_dummy'] = (graph_df['maxBase'] == destination_base).astype('int')

    # Divide destination base into quantiles
    graph_df['quantile'] = pd.qcut(graph_df[f'{var}_{destination_base}'], 10, labels=False, duplicates='drop')

    # Group by quantile and calculate averages
    grouped_quantiles = graph_df.groupby('quantile').agg({
        f'{var}_{destination_base}': 'mean', 
        'base_dummy': 'mean'
    }).reset_index()

    # Plot on the respective subplot
    ax = axes[i]
    ax.plot(grouped_quantiles['quantile'], grouped_quantiles[f'{var}_{destination_base}'], marker='o', label='Average (Projected)')
    ax.plot(grouped_quantiles['quantile'], grouped_quantiles[f'base_dummy'], marker='x', label='Average (Actual)')
    ax.set_title(f'{var}: {origin_base} -> {destination_base}, Event: {event}')
    ax.set_xlabel('Quantile (Projected)')
    ax.set_ylabel('Average')
    ax.legend()
    ax.grid(True)

# Adjust layout
plt.tight_layout()
plt.show()

In [ ]:
advances_df['to_1b'] = (advances_df['maxBase'] == 1).astype(int)
advances_df['to_2b'] = (advances_df['maxBase'] == 2).astype(int)
advances_df['to_3b'] = (advances_df['maxBase'] == 3).astype(int)
advances_df['to_home'] = (advances_df['maxBase'] == 4).astype(int)

In [ ]:
print(f"2B -> H on a Single: {advances_df.query('event_b1 == 1 and minBase_2 == 1')['base_4'].mean()}, {advances_df.query('event_b1 == 1 and minBase_2 == 1')['to_home'].mean()}")
print(f"1B -> H on a Double: {advances_df.query('event_b2 == 1 and minBase_1 == 1')['base_4'].mean()}, {advances_df.query('event_b2 == 1 and minBase_1 == 1')['to_home'].mean()}")
print(f"1B -> 3B on a Single: {advances_df.query('event_b1 == 1 and minBase_1 == 1')['base_3'].mean()}, {advances_df.query('event_b1 == 1 and minBase_1 == 1')['to_3b'].mean()}")
print(f"3B -> H on a FO: {advances_df.query('event_fo == 1 and minBase_3 == 1 and outs_pre < 2')['base_4'].mean()}, {advances_df.query('event_fo == 1 and minBase_3 == 1 and outs_pre < 2')['to_home'].mean()}")

### Model E. Attempt to Steal 2B

##### Inputs

In [ ]:
input_list = ['outs_pre', 'sba_imp', 'sb_imp']

In [ ]:
cutoff = 30
year = 2023

num_classifiers = 3 # Ensemble size
random_state = random.randint(10000,90000) 

sba_2b_model_parameters = {
    'hidden_layer_sizes': (64,32,16),
    'activation': 'relu',
    'max_iter': 100,
    'alpha': 0.00001,
    'learning_rate_init': 0.01, 
    'batch_size': 'auto',
    'random_state': random_state,
    'early_stopping': True,
    'tol': 0.00001,
    'n_iter_no_change': 20,
    'validation_fraction': 0.05
}

##### Sample

Universe: Runner on first. No runner on second. Runner on first is observation. Runner has 30 stolen base opportunities (instances of being on first base).

In [ ]:
X = df.dropna(subset=input_list).query(f'sbo > {cutoff}').query('pre_1b == 1 and pre_2b == 0 and minBase == 1').query(f'year >= {year}')[input_list]
y = df.dropna(subset=input_list).query(f'sbo > {cutoff}').query('pre_1b == 1 and pre_2b == 0 and minBase == 1').query(f'year >= {year}')['sba_2b']

##### Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

In [ ]:
X_train = torch.tensor(X_train.to_numpy(), dtype=torch.float32, device=device)
y_train = torch.tensor(y_train.to_numpy(), dtype=torch.long, device=device)
X_test  = torch.tensor(X_test.to_numpy(), dtype=torch.float32, device=device)
y_test  = torch.tensor(y_test.to_numpy(), dtype=torch.long, device=device)

##### Settings

##### Train

In [ ]:
# ------------------------
# SETTINGS / PARAMETERS
# ------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

input_size = X_train.shape[1]  # X_train for this model
output_size = 2  # binary
hidden_layers = sba_2b_model_parameters['hidden_layer_sizes']
lr = sba_2b_model_parameters['learning_rate_init']
num_epochs = sba_2b_model_parameters['max_iter']

num_classifiers = num_classifiers  # must be defined
random_seed = random_state         # must be defined

# Directory to save
save_dir = os.path.join(model_path, "M06. Base Running", todaysdate)
os.makedirs(save_dir, exist_ok=True)

model_filename = "predict_sba_2b"

# ------------------------
# TRAIN ENSEMBLE
# ------------------------
ensemble = []

for j in range(num_classifiers):
    print(f"Training classifier {j+1}/{num_classifiers}")

    seed = random_seed + 100*j
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)

    # Build model
    model = MLP(input_size, hidden_layers, output_size).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    # Train
    model.train()
    for epoch in range(num_epochs):
        optimizer.zero_grad()
        outputs = model(X_train)
        loss = criterion(outputs, y_train)
        loss.backward()
        optimizer.step()

    ensemble.append(model)

# ------------------------
# SAVE PYTORCH ENSEMBLE (.pt)
# ------------------------
torch_pt_filename = os.path.join(save_dir, f"{model_filename}.pt")
torch.save({
    'state_dicts': [m.state_dict() for m in ensemble],
    'input_size': input_size,
    'hidden_layers': hidden_layers,
    'output_size': output_size
}, torch_pt_filename)
print(f"Saved PyTorch ensemble to {torch_pt_filename}")

# ------------------------
# EXPORT TO NUMPY FOR FAST INFERENCE
# ------------------------
ensemble_numpy = []
for m in ensemble:
    state_dict = m.state_dict()
    layers = []
    linear_keys = [k for k in state_dict.keys() if "weight" in k]
    linear_keys.sort()

    for key in linear_keys:
        W = state_dict[key].cpu().numpy().T
        b = state_dict[key.replace("weight", "bias")].cpu().numpy()
        layers.append(W)
        layers.append(b)

    ensemble_numpy.append(layers)

# ------------------------
# BUILD NUMPY WRAPPER
# ------------------------
metadata = {
    "hidden_layers": hidden_layers,
    "num_classifiers": num_classifiers,
    "random_seed": random_seed,
    "training_epochs": num_epochs
}

predict_sba_2b = NumpyPredict(
    ensemble_numpy=ensemble_numpy,
    input_columns=input_list,
    classes=[0,1],
    metadata=metadata
)

# ------------------------
# SAVE WRAPPER
# ------------------------
pickle_filename = os.path.join(save_dir, f"{model_filename}.sav")
with open(pickle_filename, "wb") as f:
    pickle.dump(predict_sba_2b, f)

print(f"Saved NumpyPredict wrapper to {pickle_filename}")

##### Predict

In [ ]:
# Convert X_test tensor to DataFrame
X_test_df = pd.DataFrame(X_test.cpu().numpy(), columns=input_list)

# Preserve the original target (y_test) as a Series
y_test_series = pd.Series(y_test.cpu().numpy(), name='sba_2b', index=X_test_df.index)

# Get probability predictions from NumpyPredict wrapper
probabilities = predict_sba_2b.predict_proba(X_test_df)

# Create DataFrame for probabilities
probability_df = pd.DataFrame(probabilities, columns=['sba_2b_not', 'sba_2b_pred'], index=X_test_df.index)

# Concatenate features, target, and predicted probabilities
sba_2b_df = pd.concat([X_test_df, y_test_series, probability_df], axis=1)

##### Evaluate

In [ ]:
quantiles = 10

# Add quantiles (to examine how well predictions match actual results)
sba_2b_df['quantile'] = pd.qcut(sba_2b_df['sba_2b_pred'], quantiles, labels=False)
globals()["sba_2b_df_quantiles"] = sba_2b_df.groupby('quantile')[['sba_2b', 'sba_2b_pred']].mean().reset_index()

In [ ]:
print(sba_2b_df[['sba_2b_pred', 'sba_2b']].mean())

# Create figures
plt.plot(sba_2b_df_quantiles['quantile'], sba_2b_df_quantiles['sba_2b_pred'], color='red')
plt.plot(sba_2b_df_quantiles['quantile'], sba_2b_df_quantiles['sba_2b'], color='black')
plt.show() 

### Model F. Attempt to Steal 3B

##### Inputs

In [ ]:
input_list = ['outs_pre', 'sba_imp', 'sb_imp']

##### Settings

In [ ]:
cutoff = 30
year = 2023

num_classifiers = 3 # Ensemble size
random_state = random.randint(10000,90000) 

sba_3b_model_parameters = {
    'hidden_layer_sizes': (64,32,16),
    'activation': 'relu',
    'max_iter': 100,
    'alpha': 0.00001,
    'learning_rate_init': 0.01, 
    'batch_size': 'auto',
    'random_state': random_state,
    'early_stopping': True,
    'tol': 0.00001,
    'n_iter_no_change': 20,
    'validation_fraction': 0.05
}

##### Sample

In [ ]:
X = df.dropna(subset=input_list).query(f'sbo > {cutoff}').query('pre_2b == 1 and pre_3b == 0 and minBase == 2').query(f'year >= {year}')[input_list]
y = df.dropna(subset=input_list).query(f'sbo > {cutoff}').query('pre_2b == 1 and pre_3b == 0 and minBase == 2').query(f'year >= {year}')['sba_3b']

##### Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
X_train = torch.tensor(X_train.to_numpy(), dtype=torch.float32, device=device)
y_train = torch.tensor(y_train.to_numpy(), dtype=torch.long, device=device)
X_test  = torch.tensor(X_test.to_numpy(), dtype=torch.float32, device=device)
y_test  = torch.tensor(y_test.to_numpy(), dtype=torch.long, device=device)

##### Train

In [ ]:
# ------------------------
# SETTINGS / PARAMETERS
# ------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

input_size = X_train.shape[1]  # X_train for this model
output_size = 2  # binary classification
hidden_layers = sba_3b_model_parameters['hidden_layer_sizes']
lr = sba_3b_model_parameters['learning_rate_init']
num_epochs = sba_3b_model_parameters['max_iter']

num_classifiers = num_classifiers  # must be defined
random_seed = random_state         # must be defined

# Directory to save
save_dir = os.path.join(model_path, "M06. Base Running", todaysdate)
os.makedirs(save_dir, exist_ok=True)

model_filename = "predict_sba_3b"

# ------------------------
# TRAIN ENSEMBLE
# ------------------------
ensemble = []

for j in range(num_classifiers):
    print(f"Training classifier {j+1}/{num_classifiers}")

    seed = random_seed + 100*j
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)

    # Build model
    model = MLP(input_size, hidden_layers, output_size).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    # Train
    model.train()
    for epoch in range(num_epochs):
        optimizer.zero_grad()
        outputs = model(X_train)
        loss = criterion(outputs, y_train)
        loss.backward()
        optimizer.step()

    ensemble.append(model)

# ------------------------
# SAVE PYTORCH ENSEMBLE (.pt)
# ------------------------
torch_pt_filename = os.path.join(save_dir, f"{model_filename}.pt")
torch.save({
    'state_dicts': [m.state_dict() for m in ensemble],
    'input_size': input_size,
    'hidden_layers': hidden_layers,
    'output_size': output_size
}, torch_pt_filename)
print(f"Saved PyTorch ensemble to {torch_pt_filename}")

# ------------------------
# EXPORT TO NUMPY FOR FAST INFERENCE
# ------------------------
ensemble_numpy = []
for m in ensemble:
    state_dict = m.state_dict()
    layers = []
    linear_keys = [k for k in state_dict.keys() if "weight" in k]
    linear_keys.sort()

    for key in linear_keys:
        W = state_dict[key].cpu().numpy().T
        b = state_dict[key.replace("weight", "bias")].cpu().numpy()
        layers.append(W)
        layers.append(b)

    ensemble_numpy.append(layers)

# ------------------------
# BUILD NUMPY WRAPPER
# ------------------------
metadata = {
    "hidden_layers": hidden_layers,
    "num_classifiers": num_classifiers,
    "random_seed": random_seed,
    "training_epochs": num_epochs
}

predict_sba_3b = NumpyPredict(
    ensemble_numpy=ensemble_numpy,
    input_columns=input_list,
    classes=[0,1],
    metadata=metadata
)

# ------------------------
# SAVE WRAPPER
# ------------------------
pickle_filename = os.path.join(save_dir, f"{model_filename}.sav")
with open(pickle_filename, "wb") as f:
    pickle.dump(predict_sba_3b, f)

print(f"Saved NumpyPredict wrapper to {pickle_filename}")


##### Predict

In [ ]:
# Convert X_test tensor to DataFrame
X_test_df = pd.DataFrame(X_test.cpu().numpy(), columns=input_list)

# Preserve the original target (y_test) as a Series
y_test_series = pd.Series(y_test.cpu().numpy(), name='sba_3b', index=X_test_df.index)

# Get probability predictions from NumpyPredict wrapper
probabilities = predict_sba_3b.predict_proba(X_test_df)

# Create DataFrame for probabilities
probability_df = pd.DataFrame(probabilities, columns=['sba_3b_not', 'sba_3b_pred'], index=X_test_df.index)

# Concatenate features, target, and predicted probabilities
sba_3b_df = pd.concat([X_test_df, y_test_series, probability_df], axis=1)

##### Evaluate

In [ ]:
quantiles = 10 

# Add quantiles (to examine how well predictions match actual results)
sba_3b_df['quantile'] = pd.qcut(sba_3b_df['sba_3b_pred'], quantiles, labels=False)
globals()["sba_3b_df_quantiles"] = sba_3b_df.groupby('quantile')[['sba_3b', 'sba_3b_pred']].mean().reset_index()

In [ ]:
print(sba_3b_df[['sba_3b_pred', 'sba_3b']].mean())

# Create figures
plt.plot(sba_3b_df_quantiles['quantile'], sba_3b_df_quantiles['sba_3b_pred'], color='red')
plt.plot(sba_3b_df_quantiles['quantile'], sba_3b_df_quantiles['sba_3b'], color='black')
plt.show() 

### Model G. Steal 2B

##### Inputs

In [ ]:
input_list = ['outs_pre', 'sba_imp', 'sb_imp']

##### Settings

In [ ]:
cutoff = 30
year = 2023

num_classifiers = 3 # Ensemble size
random_state = random.randint(10000,90000) 

sb_2b_model_parameters = {
    'hidden_layer_sizes': (64,32,16),
    'activation': 'relu',
    'max_iter': 100,
    'alpha': 0.00001,
    'learning_rate_init': 0.01, 
    'batch_size': 'auto',
    'random_state': random_state,
    'early_stopping': True,
    'tol': 0.00001,
    'n_iter_no_change': 20,
    'validation_fraction': 0.05
}

##### Sample

In [ ]:
X = df.dropna(subset=input_list).query(f'sbo > {cutoff}').query('pre_1b == 1 and pre_2b == 0 and sba_2b == 1 and minBase == 1').query(f'year >= {year}')[input_list]
y = df.dropna(subset=input_list).query(f'sbo > {cutoff}').query('pre_1b == 1 and pre_2b == 0 and sba_2b == 1 and minBase == 1').query(f'year >= {year}')['sb_2b']

##### Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

In [ ]:
X_train = torch.tensor(X_train.to_numpy(), dtype=torch.float32, device=device)
y_train = torch.tensor(y_train.to_numpy(), dtype=torch.long, device=device)
X_test  = torch.tensor(X_test.to_numpy(), dtype=torch.float32, device=device)
y_test  = torch.tensor(y_test.to_numpy(), dtype=torch.long, device=device)

##### Train

In [ ]:
# ------------------------
# SETTINGS / PARAMETERS
# ------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

input_size = X_train.shape[1]  # X_train for this model
output_size = 2  # binary classification
hidden_layers = sb_2b_model_parameters['hidden_layer_sizes']
lr = sb_2b_model_parameters['learning_rate_init']
num_epochs = sb_2b_model_parameters['max_iter']

num_classifiers = num_classifiers  # must be defined
random_seed = random_state         # must be defined

# Directory to save
save_dir = os.path.join(model_path, "M06. Base Running", todaysdate)
os.makedirs(save_dir, exist_ok=True)

model_filename = "predict_sb_2b"

# ------------------------
# TRAIN ENSEMBLE
# ------------------------
ensemble = []

for j in range(num_classifiers):
    print(f"Training classifier {j+1}/{num_classifiers}")

    seed = random_seed + 100*j
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)

    # Build model
    model = MLP(input_size, hidden_layers, output_size).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    # Train
    model.train()
    for epoch in range(num_epochs):
        optimizer.zero_grad()
        outputs = model(X_train)
        loss = criterion(outputs, y_train)
        loss.backward()
        optimizer.step()

    ensemble.append(model)

# ------------------------
# SAVE PYTORCH ENSEMBLE (.pt)
# ------------------------
torch_pt_filename = os.path.join(save_dir, f"{model_filename}.pt")
torch.save({
    'state_dicts': [m.state_dict() for m in ensemble],
    'input_size': input_size,
    'hidden_layers': hidden_layers,
    'output_size': output_size
}, torch_pt_filename)
print(f"Saved PyTorch ensemble to {torch_pt_filename}")

# ------------------------
# EXPORT TO NUMPY FOR FAST INFERENCE
# ------------------------
ensemble_numpy = []
for m in ensemble:
    state_dict = m.state_dict()
    layers = []
    linear_keys = [k for k in state_dict.keys() if "weight" in k]
    linear_keys.sort()

    for key in linear_keys:
        W = state_dict[key].cpu().numpy().T
        b = state_dict[key.replace("weight", "bias")].cpu().numpy()
        layers.append(W)
        layers.append(b)

    ensemble_numpy.append(layers)

# ------------------------
# BUILD NUMPY WRAPPER
# ------------------------
metadata = {
    "hidden_layers": hidden_layers,
    "num_classifiers": num_classifiers,
    "random_seed": random_seed,
    "training_epochs": num_epochs
}

predict_sb_2b = NumpyPredict(
    ensemble_numpy=ensemble_numpy,
    input_columns=input_list,
    classes=[0,1],
    metadata=metadata
)

# ------------------------
# SAVE WRAPPER
# ------------------------
pickle_filename = os.path.join(save_dir, f"{model_filename}.sav")
with open(pickle_filename, "wb") as f:
    pickle.dump(predict_sb_2b, f)

print(f"Saved NumpyPredict wrapper to {pickle_filename}")


##### Predict

In [ ]:
# Convert X_test tensor to DataFrame
X_test_df = pd.DataFrame(X_test.cpu().numpy(), columns=input_list)

# Preserve the original target (y_test) as a Series
y_test_series = pd.Series(y_test.cpu().numpy(), name='sb_2b', index=X_test_df.index)

# Get probability predictions from NumpyPredict wrapper
probabilities = predict_sb_2b.predict_proba(X_test_df)

# Create DataFrame for probabilities
probability_df = pd.DataFrame(probabilities, columns=['sb_2b_not', 'sb_2b_pred'], index=X_test_df.index)

# Concatenate features, target, and predicted probabilities
sb_2b_df = pd.concat([X_test_df, y_test_series, probability_df], axis=1)

##### Evaluate

In [ ]:
quantiles = 10

# Add xtiles (to examine how well predictions match actual results)
sb_2b_df['quantile'] = pd.qcut(sb_2b_df['sb_2b_pred'], quantiles, labels=False)
globals()["sb_2b_df"] = sb_2b_df.groupby('quantile')[['sb_2b', 'sb_2b_pred']].mean().reset_index()

In [ ]:
print(sb_2b_df[['sb_2b_pred', 'sb_2b']].mean())

# Create figures
plt.plot(sb_2b_df['quantile'], sb_2b_df['sb_2b_pred'], color='red')
plt.plot(sb_2b_df['quantile'], sb_2b_df['sb_2b'], color='black')
plt.show() 

### Model H. Steal 3B

##### Inputs

In [ ]:
input_list = ['outs_pre', 'sba_imp', 'sb_imp']

##### Settings

In [ ]:
cutoff = 30
year = 2023

num_classifiers = 3 # Ensemble size
random_state = random.randint(10000,90000) 

sb_3b_model_parameters = {
    'hidden_layer_sizes': (64,32,16),
    'activation': 'relu',
    'max_iter': 100,
    'alpha': 0.00001,
    'learning_rate_init': 0.001, 
    'batch_size': 'auto',
    'random_state': random_state,
    'early_stopping': True,
    'tol': 0.00001,
    'n_iter_no_change': 20,
    'validation_fraction': 0.05
}

##### Sample

In [ ]:
X = df.dropna(subset=input_list).query(f'sbo > {cutoff}').query('pre_2b == 1 and pre_3b == 0 and sba_3b == 1 and minBase == 2').query(f'year >= {year}')[input_list]
y = df.dropna(subset=input_list).query(f'sbo > {cutoff}').query('pre_2b == 1 and pre_3b == 0 and sba_3b == 1 and minBase == 2').query(f'year >= {year}')['sb_3b']

##### Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1000000)

In [ ]:
X_train = torch.tensor(X_train.to_numpy(), dtype=torch.float32, device=device)
y_train = torch.tensor(y_train.to_numpy(), dtype=torch.long, device=device)
X_test  = torch.tensor(X_test.to_numpy(), dtype=torch.float32, device=device)
y_test  = torch.tensor(y_test.to_numpy(), dtype=torch.long, device=device)

##### Train

In [ ]:
# ------------------------
# SETTINGS / PARAMETERS
# ------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

input_size = X_train.shape[1]  # X_train for this model
output_size = 2  # binary classification
hidden_layers = sb_3b_model_parameters['hidden_layer_sizes']
lr = sb_3b_model_parameters['learning_rate_init']
num_epochs = sb_3b_model_parameters['max_iter']

num_classifiers = num_classifiers  # must be defined
random_seed = random_state         # must be defined

# Directory to save
save_dir = os.path.join(model_path, "M06. Base Running", todaysdate)
os.makedirs(save_dir, exist_ok=True)

model_filename = "predict_sb_3b"

# ------------------------
# TRAIN ENSEMBLE
# ------------------------
ensemble = []

for j in range(num_classifiers):
    print(f"Training classifier {j+1}/{num_classifiers}")

    seed = random_seed + 100*j
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)

    # Build model
    model = MLP(input_size, hidden_layers, output_size).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    # Train
    model.train()
    for epoch in range(num_epochs):
        optimizer.zero_grad()
        outputs = model(X_train)
        loss = criterion(outputs, y_train)
        loss.backward()
        optimizer.step()

    ensemble.append(model)

# ------------------------
# SAVE PYTORCH ENSEMBLE (.pt)
# ------------------------
torch_pt_filename = os.path.join(save_dir, f"{model_filename}.pt")
torch.save({
    'state_dicts': [m.state_dict() for m in ensemble],
    'input_size': input_size,
    'hidden_layers': hidden_layers,
    'output_size': output_size
}, torch_pt_filename)
print(f"Saved PyTorch ensemble to {torch_pt_filename}")

# ------------------------
# EXPORT TO NUMPY FOR FAST INFERENCE
# ------------------------
ensemble_numpy = []
for m in ensemble:
    state_dict = m.state_dict()
    layers = []
    linear_keys = [k for k in state_dict.keys() if "weight" in k]
    linear_keys.sort()

    for key in linear_keys:
        W = state_dict[key].cpu().numpy().T
        b = state_dict[key.replace("weight", "bias")].cpu().numpy()
        layers.append(W)
        layers.append(b)

    ensemble_numpy.append(layers)

# ------------------------
# BUILD NUMPY WRAPPER
# ------------------------
metadata = {
    "hidden_layers": hidden_layers,
    "num_classifiers": num_classifiers,
    "random_seed": random_seed,
    "training_epochs": num_epochs
}

predict_sb_3b = NumpyPredict(
    ensemble_numpy=ensemble_numpy,
    input_columns=input_list,
    classes=[0,1],
    metadata=metadata
)

# ------------------------
# SAVE WRAPPER
# ------------------------
pickle_filename = os.path.join(save_dir, f"{model_filename}.sav")
with open(pickle_filename, "wb") as f:
    pickle.dump(predict_sb_3b, f)

print(f"Saved NumpyPredict wrapper to {pickle_filename}")


##### Predict

In [ ]:
# Convert X_test tensor to DataFrame
X_test_df = pd.DataFrame(X_test.cpu().numpy(), columns=input_list)

# Preserve the original target (y_test) as a Series
y_test_series = pd.Series(y_test.cpu().numpy(), name='sb_3b', index=X_test_df.index)

# Get probability predictions from NumpyPredict wrapper
probabilities = predict_sb_3b.predict_proba(X_test_df)

# Create DataFrame for probabilities
probability_df = pd.DataFrame(probabilities, columns=['sb_3b_not', 'sb_3b_pred'], index=X_test_df.index)

# Concatenate features, target, and predicted probabilities
sb_3b_df = pd.concat([X_test_df, y_test_series, probability_df], axis=1)

##### Evaluate

In [ ]:
quantiles = 5

# Add xtiles (to examine how well predictions match actual results)
sb_3b_df['quantile'] = pd.qcut(sb_3b_df['sb_3b_pred'], quantiles, labels=False)
globals()["sb_3b_df"] = sb_3b_df.groupby('quantile')[['sb_3b', 'sb_3b_pred']].mean().reset_index()

In [ ]:
print(sb_3b_df[['sb_3b_pred', 'sb_3b']].mean())

# Create figures
plt.plot(sb_3b_df['quantile'], sb_3b_df['sb_3b_pred'], color='red')
plt.plot(sb_3b_df['quantile'], sb_3b_df['sb_3b'], color='black')
plt.show()

Note: Stolen base models are restricted to 2023 and later to account for changes to base sizes and pick off rules. This reduces sample size, but should result in more realistic average rates.